Bloco 1: Instalação

In [1]:
!pip install -q openai-whisper yfinance gTTS
import whisper
import yfinance as yf
from gtts import gTTS
from IPython.display import Audio, display
import os

# Carrega o modelo de transcrição gratuito
model_whisper = whisper.load_model("base")
print("✅ Tudo pronto e gratuito!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 103MiB/s]


✅ Tudo pronto e gratuito!


Bloco 2: As Funções do Projeto

In [3]:
def identificar_ativo(texto_usuario):
    texto = texto_usuario.lower()
    if "dolar" in texto or "dólar" in texto: return "USDBRL=X"
    if "ibov" in texto or "ibovespa" in texto: return "^BVSP"
    if "petr4" in texto: return "PETR4.SA"
    if "vale3" in texto: return "VALE3.SA"
    return None

def get_preco(ticker):
    acao = yf.Ticker(ticker)
    dados = acao.history(period="1d")
    return round(dados['Close'].iloc[-1], 2) if not dados.empty else 0.0

def falar(texto):
    tts = gTTS(text=texto, lang="pt", slow=False)
    tts.save("res.mp3")
    display(Audio("res.mp3", autoplay=True))

Bloco 3: O Gravador

In [5]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Bloco JavaScript para captura de áudio
RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record = async (time) => {
  // Configuração para microfones analógicos (Mono e sem filtros que abafam o som)
  const stream = await navigator.mediaDevices.getUserMedia({
    audio: {
      channelCount: 1,
      echoCancellation: false,
      noiseSuppression: false,
      autoGainControl: true
    }
  })

  const recorder = new MediaRecorder(stream)
  const chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)

  return new Promise(async resolve => {
    recorder.onstop = async () => {
      const blob = new Blob(chunks)
      const text = await b2text(blob)
      resolve(text)
    }
    recorder.stop()
  })
}
"""

def record_audio(sec=5):
    # Injeta o JavaScript na saída do Colab
    display(Javascript(RECORD))

    # Executa a função JS e aguarda o retorno (Base64)
    print(f"Gravando por {sec} segundos... Fale agora!")
    js_result = output.eval_js(f'record({sec * 1000})')

    # Decodifica o Base64 para binário (WAV)
    audio_bytes = b64decode(js_result.split(',')[1])

    file_name = 'request_audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio_bytes)

    print("Gravação finalizada!")
    return f'/content/{file_name}'

# --- EXECUÇÃO ---

# 1. Chama a função de gravação
record_file = record_audio(sec=5)

# 2. Exibe o player para teste
display(Audio(record_file, autoplay=False))

<IPython.core.display.Javascript object>

Gravando por 5 segundos... Fale agora!
Gravação finalizada!


Bloco 4: Integração Final

In [7]:
import os

# 1. Transcreve sua voz (Usando o arquivo que o seu gravador acabou de criar)
caminho_do_audio = 'request_audio.wav'

print("🔍 Entendendo o que você disse...")

if os.path.exists(caminho_do_audio):
    # O Whisper lê o arquivo gerado pelo seu bloco de JavaScript
    result = model_whisper.transcribe(caminho_do_audio, fp16=False, language="pt")
    user_text = result["text"]
    print(f"🗣️ Você disse: {user_text}")

    # 2. Lógica de Identificação do Ativo
    ativo = identificar_ativo(user_text)

    if ativo:
        # 3. Busca o preço no Yahoo Finance
        preco = get_preco(ativo)

        # 4. Cria a resposta e fala
        nome_exibicao = "Dólar" if "USDBRL" in ativo else "Ibovespa" if "^BVSP" in ativo else ativo
        resposta = f"O preço atual de {nome_exibicao} é {preco} reais."

        print(f"🤖 Resposta: {resposta}")
        falar(resposta)
    else:
        msg = "Não identifiquei o ativo. Tente falar 'Dólar', 'Petr4' ou 'Vale3'."
        print(msg)
        falar(msg)
else:
    print(f"❌ Erro: O arquivo {caminho_do_audio} não foi encontrado. Rode o bloco de gravação acima primeiro!")

🔍 Entendendo o que você disse...
🗣️ Você disse:  Cotação do dólar
🤖 Resposta: O preço atual de Dólar é 5.19 reais.
